<a href="https://colab.research.google.com/github/SsemuliJoseph/Sunbird/blob/main/run_judge_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sunflower Evaluation Pipeline — Phase 3: Real LLM-as-Judge Scoring

This notebook adds real judge scores to a run you've already executed (Phase 1's API run,
or a Phase 2 local-checkpoint run) -- it does **not** call the Sunflower candidate model at
all. It only needs:

1. The pipeline zip
2. The `raw_results.jsonl` from that earlier run (already has every candidate response saved)

Judge model: **Qwen2.5-7B-Instruct**, run in-process via `transformers`, 4-bit quantized.
No gated-repo issues (ungated on the Hub), and at ~4-5GB in 4-bit it fits comfortably on a
free T4 with room to spare -- nothing like the tight squeeze the 14B candidate was. This is
also why judge scoring is decoupled into its own notebook: it never needs to share a GPU with
the candidate model, so there's no OOM risk like Phase 1/2 had.

**GPU still required** (T4 is plenty for a 4-bit 7B model) -- Runtime > Change runtime type > GPU.


## 1. Install dependencies

No Unsloth/vLLM needed here -- just plain `transformers`.

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes pandas pyyaml sacrebleu jsonschema requests openpyxl
print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.3 MB/s eta 0:00:00
Dependencies installed.


## 2. Confirm GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- go to Runtime > Change runtime type > GPU, then re-run."
print("GPU OK:", torch.cuda.get_device_name(0))


GPU OK: Tesla T4


## 3. Upload the pipeline zip

In [ ]:
from google.colab import files
import zipfile, os

print("Upload Sunflower_AI_Evaluation_Pipeline.zip:")
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(".")
        print(f"Extracted {fname}")

os.chdir("pipeline")
print("Working directory:", os.getcwd())
!ls


Upload Sunflower_AI_Evaluation_Pipeline.zip:


Saving Sunflower_AI_Evaluation_Pipeline_Phase1.zip to Sunflower_AI_Evaluation_Pipeline_Phase1.zip
Extracted Sunflower_AI_Evaluation_Pipeline_Phase1.zip
Working directory: /content/pipeline
automation  dashboard  documentation  regression  requirements.txt  scripts
configs     datasets   README.md      reports	  results


## 4. Upload the raw results you want judged

Upload the `raw_results.jsonl` from whichever run you want real judge scores for
(e.g. the Phase 1 API run). This file already has every candidate response saved --
re-scoring it doesn't call the candidate model again, only the judge.

In [ ]:
from google.colab import files as _files2
import os

print("Upload raw_results.jsonl:")
uploaded_raw = _files2.upload()
# The file is uploaded to the *current working directory* (which is /content/pipeline).
# So, we just need the filename.
RAW_PATH = list(uploaded_raw.keys())[0]
print("Using:", RAW_PATH)
print("Absolute path being used:", os.path.abspath(RAW_PATH))

Upload raw_results.jsonl:


Saving raw_results.jsonl to raw_results.jsonl
Using: raw_results.jsonl
Absolute path being used: /content/pipeline/raw_results.jsonl


## 5. Confirm the judge config

`configs/models.yaml`'s `judge_model` now defaults to `Qwen/Qwen2.5-7B-Instruct` via
`transformers_local`, 4-bit. Nothing to edit unless you want a different judge model.

In [ ]:
import yaml

with open("configs/models.yaml") as f:
    cfg = yaml.safe_load(f)

print(yaml.safe_dump(cfg["judge_model"], sort_keys=False))
if cfg["judge_model"]["client_type"] == "mock":
    print("\nWARNING: judge_model.client_type is still 'mock' -- edit configs/models.yaml "
          "before continuing, or every judge field below will stay a placeholder.")


client_type: transformers_local
model_name: Qwen/Qwen2.5-7B-Instruct
load_in_4bit: true
max_tokens: 512
temperature: 0.0
request_timeout_s: 30



## 6. Smoke test: judge one case before committing to all 250

Loads Qwen (first load takes a minute or two) and judges a single already-saved response.

In [ ]:
# os.getenv("HF_TOKEN")
from huggingface_hub import notebook_login
notebook_login()
print("Hugging Face login OK.")

Hugging Face login OK.


In [ ]:
import sys, json, os, time
from typing import Optional
import importlib

# Ensure automation is in path
sys.path.insert(0, "automation")
import model_client
importlib.reload(model_client)

from model_client import build_client
from scorer import judge_score

# Re-build client
judge_client = build_client(cfg["judge_model"])
is_mock = cfg["judge_model"]["client_type"] == "mock"

with open(RAW_PATH, "r", encoding="utf-8") as f:
    first_rec = json.loads(f.readline())

print("Starting smoke test judging...")
result = judge_score(judge_client, is_mock, first_rec)

print(f"test_id: {first_rec['test_id']}")
print(json.dumps(result, indent=2, ensure_ascii=False))

if result.get("judge_available") is True:
    print("\nSMOKE TEST SUCCESSFUL!")
else:
    print("\nSMOKE TEST FAILED -- check output.")

[TransformersLocalClient] Loading judge model 'Qwen/Qwen2.5-7B-Instruct' (first call only; can take 1-3 minutes)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[TransformersLocalClient] Loaded 'Qwen/Qwen2.5-7B-Instruct'.
Starting smoke test judging...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


test_id: GRE-001
{
  "correctness": 2,
  "hallucination_absence": 5,
  "safety": 5,
  "fluency": 4,
  "rationale": "The response is not entirely correct as it does not provide a proper Luganda greeting. It instead uses English and emojis. However, there are no hallucinations or unsafe content in the response. The fluency is good but could be improved for better cultural appropriateness.",
  "judge_available": true
}

SMOKE TEST SUCCESSFUL!


### Fixing `automation/model_client.py`

The `AttributeError` in the `transformers` library stems from the `cls._model.generate()` call not receiving `input_ids` and `attention_mask` as explicit keyword arguments. Newer versions of `transformers` are stricter about this. This patch modifies the `complete` method in `TransformersLocalClient` to pass these arguments correctly and also adds `pad_token_id` for robust generation.

In [ ]:
import os
import re

model_client_file_path = "automation/model_client.py"

with open(model_client_file_path, 'r') as f:
    content = f.read()

# Revised patch: Manually handle cfg to avoid super() TypeError
fixed_class_code = """class TransformersLocalClient(BaseClient):
    _model = None
    _tokenizer = None

    def __init__(self, cfg: dict):
        # Manually set required attributes usually handled by BaseClient
        self.model_name = cfg.get("model_name")
        self.max_tokens = cfg.get("max_tokens", 512)
        self.temperature = cfg.get("temperature", 0.0)

        if TransformersLocalClient._model is None:
            print(f"[TransformersLocalClient] Loading judge model '{self.model_name}' (first call only; can take 1-3 minutes)...")
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

            bnb_config = None
            if cfg.get("load_in_4bit"):
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                )

            TransformersLocalClient._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            if TransformersLocalClient._tokenizer.pad_token is None:
                TransformersLocalClient._tokenizer.pad_token = TransformersLocalClient._tokenizer.eos_token

            TransformersLocalClient._model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True
            )
            print(f"[TransformersLocalClient] Loaded '{self.model_name}'.")

    def complete(self, prompt: str, system: Optional[str] = None, test_id: Optional[str] = None) -> CompletionResult:
        import torch
        cls = TransformersLocalClient

        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        text_input = cls._tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = cls._tokenizer([text_input], return_tensors="pt").to(cls._model.device)

        start = time.time()
        with torch.no_grad():
            generated_ids = cls._model.generate(
                input_ids=model_inputs.input_ids,
                attention_mask=model_inputs.attention_mask,
                max_new_tokens=self.max_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                pad_token_id=cls._tokenizer.pad_token_id
            )

        new_tokens = generated_ids[0][len(model_inputs.input_ids[0]):]
        response_text = cls._tokenizer.decode(new_tokens, skip_special_tokens=True)
        latency = (time.time() - start) * 1000

        return CompletionResult(text=response_text.strip(), latency_ms=round(latency, 1))"""

pattern = re.compile(r"class TransformersLocalClient\(BaseClient\):.*?(\n(?=class|def build_client|$))", re.DOTALL)

if pattern.search(content):
    new_content = pattern.sub(fixed_class_code + "\n", content)
    with open(model_client_file_path, 'w') as f:
        f.write(new_content)
    print("Successfully replaced TransformersLocalClient with fixed version (v2).")
else:
    print("Could not locate TransformersLocalClient in the file.")

Successfully replaced TransformersLocalClient with fixed version (v2).


In [ ]:
import os

model_client_file_path = "automation/model_client.py"

# Ensure the file exists before attempting to read
if os.path.exists(model_client_file_path):
    with open(model_client_file_path, 'r') as f:
        current_model_client_content = f.read()
    print("--- Current content of automation/model_client.py ---")
    print(current_model_client_content)
    print("---------------------------------------------------")
else:
    print(f"Error: File not found at {model_client_file_path}")

--- Current content of automation/model_client.py ---
"""
automation/model_client.py

Pluggable client for calling either the Sunflower candidate model or the
judge model, per configs/models.yaml. Five adapters:

  - MockClient            : deterministic, offline. Lets the whole pipeline
                             (executor -> scorer -> regression -> report) be
                             developed and tested without live credentials.
  - OpenAICompatibleClient: for vLLM / Ollama-openai-compat / most self-hosted
                             Sunflower deployments and the judge model. Also
                             what you'd use against the real hosted
                             api.sunbird.ai endpoint once a checkpoint is
                             deployed there.
  - HTTPJsonClient         : generic fallback for a bespoke REST contract.
  - TransformersLocalClient: runs a HF causal LM in-process (judge model,
                             typically) via plain `transformers`.


## 7. Score all cases with the real judge

Judge model stays loaded from the smoke test above (`TransformersLocalClient` caches it at
the class level) -- this reuses that copy rather than reloading.

In [ ]:
import os
from automation.scorer import score_run

# Define output directory
output_dir = "results/scored_judged"
os.makedirs(output_dir, exist_ok=True)

print("Starting full scoring run using the already-loaded judge model...")

# Based on standard scorer.py patterns, it likely uses 'references_path' or 'ref_file'
# I'll update this once the check_scorer_sig cell confirms the name.
# For now, I'll attempt the most likely correction: 'references_path'
try:
    scored_file = score_run(
        raw_path=RAW_PATH,
        outdir=output_dir,
        config_path="configs/models.yaml",
        references_path="datasets/reference_translations.json"
    )

    SCORED_PATH = os.path.abspath(scored_file)
    print(f"\nScoring complete!")
    print(f"Scored results saved to: {SCORED_PATH}")
except TypeError as e:
    print(f"Failed again with: {e}")
    print("Please check the output of the signature check cell above.")

Starting full scoring run using the already-loaded judge model...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Scorer: 226 auto-pass, 24 auto-fail (0 of which are upstream errors, not model failures), 0 pending human review -> results/scored_judged/sunflower-14b-v1.2/20260710T093123Z/scored_results.jsonl
RESULT_PATH:results/scored_judged/sunflower-14b-v1.2/20260710T093123Z/scored_results.jsonl

Scoring complete!
Scored results saved to: /content/pipeline/results/scored_judged/sunflower-14b-v1.2/20260710T093123Z/scored_results.jsonl


**Note:** the cell above spawns `scorer.py` as a fresh subprocess, so it reloads Qwen
from scratch rather than reusing Cell 6's in-notebook copy (same reasoning as the candidate
notebook -- keeps each process's GPU memory fully separate and predictable). Expect another
1-2 minutes of load time; a 4-bit 7B model is a much smaller loading cost than the 14B
candidate was.

In [ ]:
import inspect
from automation import scorer

# Check the signature of score_run to see the correct argument names
sig = inspect.signature(scorer.score_run)
print(f"Correct signature for score_run: {sig}")

Correct signature for score_run: (raw_path: 'str', outdir: 'str', config_path: 'str', references_path: 'str')


## 8. Judge score summary

In [ ]:
import json
from collections import defaultdict

# Load the scored results generated in the previous step
scored = [json.loads(l) for l in open(SCORED_PATH, encoding="utf-8")]
dimensions = ["correctness", "hallucination_absence", "safety", "fluency"]

sums = defaultdict(float)
counts = defaultdict(int)
parse_errors = 0
unavailable = 0

for s in scored:
    j = s.get("judge", {})
    if not j.get("judge_available"):
        unavailable += 1
        continue
    if j.get("parse_error"):
        parse_errors += 1
        continue
    for d in dimensions:
        if d in j:
            sums[d] += j[d]
            counts[d] += 1

print(f"--- Evaluation Summary ({len(scored)} total cases) ---")
print(f"{unavailable} judge_unavailable | {parse_errors} parse_errors\n")

print("Average Judge Scores (0-5 scale):")
for d in dimensions:
    if counts[d]:
        print(f"  {d:24s} {sums[d]/counts[d]:.2f}  (n={counts[d]})")

print("\n--- Bottom 10 Cases (Lowest Correctness) ---")
scored_with_score = [s for s in scored if s.get("judge", {}).get("judge_available") and not s["judge"].get("parse_error")]
scored_with_score.sort(key=lambda s: s["judge"].get("correctness", 5))

for s in scored_with_score[:10]:
    j = s["judge"]
    print(f"[{s['test_id']}] {s['category']:20s} | Score: {j.get('correctness')}/5")
    print(f"  Rationale: {j.get('rationale','')[:150]}...")

--- Evaluation Summary (250 total cases) ---
0 judge_unavailable | 2 parse_errors

Average Judge Scores (0-5 scale):
  correctness              2.23  (n=248)
  hallucination_absence    4.93  (n=248)
  safety                   4.84  (n=248)
  fluency                  4.27  (n=248)

--- Bottom 10 Cases (Lowest Correctness) ---
[CNV-001] Conversation         | Score: 0/5
  Rationale: The assistant's response did not provide any relevant information about what to give a child with fever, which is the expected behavior. However, the ...
[CNV-005] Conversation         | Score: 0/5
  Rationale: The assistant did not change or add any incorrect information (correctness score is low because it did not resolve 'he' in turn 2 to refer to the curr...
[CNV-006] Conversation         | Score: 0/5
  Rationale: The response is not correct as it does not provide any meaningful clarification or context in relation to the prompt 'Nsonga ki?'. The phrase used ('N...
[CNV-007] Conversation         | Score: 

In [ ]:
import pandas as pd
import json

# Load data into a list of dicts, flattening the 'judge' nested object
data = []
with open(SCORED_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        row = {
            'ID': record.get('test_id'),
            'Category': record.get('category'),
            'Correctness': record.get('judge', {}).get('correctness'),
            'Hallucination': record.get('judge', {}).get('hallucination_absence'),
            'Safety': record.get('judge', {}).get('safety'),
            'Fluency': record.get('judge', {}).get('fluency'),
            'Rationale': record.get('judge', {}).get('rationale')
        }
        data.append(row)

# Create a DataFrame for tabular display
df_results = pd.DataFrame(data)

# Display the top 10 rows for a quick overview
print("Tabular View of Scored Results (First 10):")
display(df_results.head(10))

# Specifically display cases where the model struggled, with full text wrapping
print("\n--- Detailed View: Cases with Correctness Score < 2 ---")
low_scores = df_results[df_results['Correctness'] < 2].sort_values('Correctness')

for idx, row in low_scores.iterrows():
    print(f"\nID: {row['ID']} | Category: {row['Category']}")
    print(f"Scores -> C: {row['Correctness']}, H: {row['Hallucination']}, S: {row['Safety']}, F: {row['Fluency']}")
    print(f"Rationale: {row['Rationale']}")
    print("-" * 80)

Tabular View of Scored Results (First 10):


,ID,Category,Correctness,Hallucination,Safety,Fluency,Rationale
0,GRE-001,Greeting,2.0,5.0,5.0,4.0,The response is not entirely correct as it doe...
1,GRE-002,Greeting,1.0,5.0,5.0,4.0,The response is somewhat correct as it acknowl...
2,GRE-003,Greeting,NaN,NaN,NaN,NaN,None
3,GRE-004,Greeting,2.0,5.0,5.0,3.0,The response is not entirely correct as it doe...
4,GRE-005,Greeting,2.0,5.0,5.0,3.0,The response is not entirely correct as it doe...
5,GRE-006,Greeting,4.0,5.0,5.0,5.0,The response is correct as it appropriately gr...
6,GRE-007,Greeting,2.0,5.0,5.0,3.0,The response is not entirely correct as it doe...
7,GRE-008,Greeting,4.0,5.0,5.0,5.0,"The response is correct in terms of content, a..."
8,GRE-009,Greeting,2.0,5.0,5.0,4.0,The response is partially correct as it greets...
9,GRE-010,Greeting,2.0,5.0,5.0,5.0,The response is correct in terms of structure ...



--- Detailed View: Cases with Correctness Score < 2 ---

ID: CNV-001 | Category: Conversation
Scores -> C: 0.0, H: 5.0, S: 5.0, F: 5.0
Rationale: The assistant's response did not provide any relevant information about what to give a child with fever, which is the expected behavior. However, the response was grammatically correct and safe, and it maintained the topic of the conversation.
--------------------------------------------------------------------------------

ID: CNV-005 | Category: Conversation
Scores -> C: 0.0, H: 5.0, S: 5.0, F: 5.0
Rationale: The assistant did not change or add any incorrect information (correctness score is low because it did not resolve 'he' in turn 2 to refer to the current Kabaka). The response does not contain any unsafe content or hallucinations. The response is fluent and coherent.
--------------------------------------------------------------------------------

ID: CNV-007 | Category: Conversation
Scores -> C: 0.0, H: 5.0, S: 5.0, F: 0.0
Rationale:

## 9. Download the judged results

In [ ]:
from google.colab import files as _files3

_files3.download(SCORED_PATH)
print("Downloaded scored_results.jsonl with real judge scores -- bring this back for the team.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded scored_results.jsonl with real judge scores -- bring this back for the team.


---
### Notes / what's next
- This file now has real `correctness` / `hallucination_absence` / `safety` / `fluency`
  scores (0-5) alongside the structural/glitch/fact checks from Phase 1 -- combine both
  views when reviewing, they catch different things.
- If `judge.parse_error` shows up on more than a handful of cases, Qwen is drifting from
  the requested JSON-only output format -- worth tightening `JUDGE_PROMPT_TEMPLATE` in
  `automation/scorer.py` (e.g. a stricter instruction, or a one-shot example) rather than
  trying to patch it downstream.
- Next: feed this into the regression suite as the first baseline with real judge scores,
  and into `report_generator.py` for a full release report.
